In [ ]:
# transformers must be installed from github
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q torch torchvision Pillow tqdm accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch

In [ ]:
print(f"CUDA available : {torch.cuda.is_available()}")

CUDA available : True


In [ ]:
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9

    print(f"VRAM: {total_vram:.1f} GB")

    if total_vram < 8:
        print("Less than 8 GB VRAM -> switch to T4 in Runtime settings!")
    else:
        print("VRAM looks good.")
else:
    print("No GPU found -> go to Runtime -> Change runtime type -> T4 GPU")

GPU: Tesla T4
VRAM: 15.6 GB
VRAM looks good.


# Load the model, you should expect around a 2min waiting time

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

In [ ]:
MODEL_PATH = "zai-org/GLM-OCR"

print("Loading processor...")

processor = AutoProcessor.from_pretrained(MODEL_PATH)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

# check real VRAM used after loading
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    print(f"\n Model is fully loaded. GPU VRAM used: {used:.2f} GB")

Loading processor...


model.safetensors:   0%|          | 0.00/2.65G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]


 Model is fully loaded. GPU VRAM used: 2.22 GB


In [1]:
from PIL import Image
from pathlib import Path
import requests
from io import BytesIO

TASK_PROMPTS = {
    "text"    : "Text Recognition:",

    "formula" : "Formula Recognition:",

    "table"   : "Table Recognition:",
}

def load_image(source):
    if isinstance(source, Image.Image):
        return source

    if isinstance(source, str) and source.startswith("http"):
        resp = requests.get(source, timeout=10)

        return Image.open(BytesIO(resp.content)).convert("RGB")

    return Image.open(source).convert("RGB")


def ocr_single(image_source, task="text", max_new_tokens=8192):
    prompt = TASK_PROMPTS.get(task, task)

    image  = load_image(image_source)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt},
        ],
    }]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    inputs.pop("token_type_ids", None)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )

    output = processor.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=False,
    )
    return output.strip()


print("ocr_single() completed")

ocr_single() completed


In [3]:
import json
from tqdm.auto import tqdm
from datetime import datetime

def batch_ocr(
    image_sources,
    task="text",
    output_jsonl=None,
    clear_cache_every=10,
):
    results = []
    writer  = open(output_jsonl, "w") if output_jsonl else None

    try:
        for idx, src in enumerate(tqdm(image_sources, desc="OCR")):
            record = {"index": idx, "source": str(src), "text": None, "error": None}
            try:
                record["text"] = ocr_single(src, task=task)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                record["error"] = "try reducing image resolution)"
                print(f"\n OOM on image {idx}: {src}")
            except Exception as e:
                record["error"] = str(e)
                print(f"\n error on image {idx}: {e}")

            results.append(record)

            if writer:
                writer.write(json.dumps(record, ensure_ascii=False) + "\n")
                writer.flush()

            if (idx + 1) % clear_cache_every == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
    finally:
        if writer:
            writer.close()

    success = sum(1 for r in results if r["error"] is None)
    print(f"\n Done: {success}/{len(results)} images processed successfully.")
    return results


print("batch_ocr() completed")

batch_ocr() completed


> Change the Folder location to your folder



In [4]:
import glob

FOLDER = "/content/images"

EXTENSIONS = ("*.png", "*.jpg", "*.jpeg", "*.tiff", "*.webp")

image_files = []

for ext in EXTENSIONS:
    image_files.extend(glob.glob(f"{FOLDER}/{ext}"))

image_files.sort()

print(f"Found {len(image_files)} images in {FOLDER}")

if image_files:
    results = batch_ocr(
        image_files,
        task="text",
        output_jsonl="/content/ocr_results.jsonl",
        clear_cache_every=10,
    )

Found 0 images in /content/images
